# 04.08 Signature and Stamp Screening

**Phase 04 — Document Processing & OCR (Track A: Document Screening)**  
**Objective:** Localize candidate signature and official stamp/seal regions using trained YOLO weights (when available) with an explicit OpenCV color/ink-density contour screening fallback, score region presence confidence, flag uncertain regions, and export structured results for downstream validation.

> **Governance Disclaimer:** Detecting a signature-like or stamp-like region indicates **presence only** — it never certifies that a signature or institutional seal is genuine. Institutional signatory verification is mandatory.


### 1. Environment Setup & Detector Discovery


In [ ]:
import os
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

SIG_STAMP_OUT_DIR = paths.interim_doc_proc / 'signature_stamp_screening'
SIG_STAMP_OUT_DIR.mkdir(parents=True, exist_ok=True)
SIG_STAMP_REPORT_JSON = SIG_STAMP_OUT_DIR / 'signature_stamp_screening_results.json'

YOLO_WEIGHT_CANDIDATES = [
    paths.root / 'experiments' / 'models' / 'trained' / 'yolo' / 'ofphil_yolo_detector' / 'weights' / 'best.pt',
    paths.sig_stamp_model_dir / 'best.pt',
]
YOLO_WEIGHTS_PATH = next((p for p in YOLO_WEIGHT_CANDIDATES if p.exists()), None)

print(f"Project Root      : {paths.root}")
print(f"YOLO Weights Found: {YOLO_WEIGHTS_PATH}")
print(f"Report Output     : {SIG_STAMP_REPORT_JSON}")


### 2. Signature & Stamp Presence Screening Pipeline


In [ ]:
def detect_stamp_and_signature_opencv(bgr_img: np.ndarray) -> dict:
    """
    Deterministic OpenCV presence screener:
    - Stamps: high-saturation blue/red/purple ink clusters in HSV space.
    - Signatures: dark connected stroke clusters in the lower 40% of the document.
    """
    h, w = bgr_img.shape[:2]
    hsv = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)

    # Blue/Purple & Red stamp ink masks
    blue_mask = cv2.inRange(hsv, np.array([95, 55, 40]), np.array([150, 255, 255]))
    red_m1 = cv2.inRange(hsv, np.array([0, 65, 40]), np.array([12, 255, 255]))
    red_m2 = cv2.inRange(hsv, np.array([168, 65, 40]), np.array([180, 255, 255]))
    stamp_mask = cv2.bitwise_or(blue_mask, cv2.bitwise_or(red_m1, red_m2))
    stamp_mask = cv2.morphologyEx(stamp_mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))

    stamp_boxes = []
    cnts, _ = cv2.findContours(stamp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_stamp_area = max(300, int(0.0015 * h * w))
    for c in cnts:
        x, y, bw, bh = cv2.boundingRect(c)
        if bw * bh >= min_stamp_area and 0.3 <= (bw / max(1, bh)) <= 3.5:
            density = float(stamp_mask[y:y+bh, x:x+bw].mean() / 255.0)
            stamp_boxes.append({'bbox': [int(x), int(y), int(bw), int(bh)], 'confidence': round(min(0.95, 0.45 + density), 3)})

    # Lower-page signature candidate strokes (grayscale adaptive threshold)
    y0 = int(h * 0.55)
    lower_gray = cv2.cvtColor(bgr_img[y0:, :], cv2.COLOR_BGR2GRAY)
    inv_bin = cv2.adaptiveThreshold(lower_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 10)
    dilated = cv2.dilate(inv_bin, np.ones((5, 15), np.uint8), iterations=1)
    s_cnts, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    sig_boxes = []
    for c in s_cnts:
        x, y, bw, bh = cv2.boundingRect(c)
        ar = bw / float(max(1, bh))
        if bw >= int(w * 0.08) and bh >= 18 and 1.8 <= ar <= 8.0:
            ink_ratio = float(inv_bin[y:y+bh, x:x+bw].mean() / 255.0)
            if 0.03 <= ink_ratio <= 0.45:
                sig_boxes.append({'bbox': [int(x), int(y + y0), int(bw), int(bh)], 'confidence': round(min(0.90, 0.40 + ink_ratio * 1.5), 3)})

    sig_boxes = sorted(sig_boxes, key=lambda d: d['confidence'], reverse=True)[:5]
    stamp_boxes = sorted(stamp_boxes, key=lambda d: d['confidence'], reverse=True)[:5]
    return {'signature_candidates': sig_boxes, 'stamp_candidates': stamp_boxes, 'backend': 'OPENCV_HSV_CONTOUR'}

def screen_document_sig_stamp(image_path: Path) -> tuple:
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Unable to read image: {image_path}")
    det = detect_stamp_and_signature_opencv(img)
    n_sig = len(det['signature_candidates'])
    n_st = len(det['stamp_candidates'])
    max_sig_c = max([d['confidence'] for d in det['signature_candidates']], default=0.0)
    max_st_c = max([d['confidence'] for d in det['stamp_candidates']], default=0.0)
    status = 'CANDIDATES_DETECTED_REVIEW_REQUIRED' if (n_sig > 0 or n_st > 0) else 'NO_CLEAR_SIGNATURE_OR_STAMP_DETECTED'
    record = {
        'filename': image_path.name,
        'backend': det['backend'],
        'signatures_detected': n_sig,
        'max_signature_confidence': max_sig_c,
        'stamps_detected': n_st,
        'max_stamp_confidence': max_st_c,
        'screening_status': status,
        'manual_institutional_verification_required': True,
        'disclaimer': 'Presence screening only; never certifies signature or stamp authenticity.',
    }
    return record, img, det

sample_imgs = []
if paths.doc_inventory_manifest.exists():
    df_inv = pd.read_csv(paths.doc_inventory_manifest)
    for _, r in df_inv.head(20).iterrows():
        p = paths.resolve_sample_path(str(r['relative_path']))
        if p.exists() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            sample_imgs.append(p)

records, vis_sample = [], None
for ip in sample_imgs:
    rec, bgr, det = screen_document_sig_stamp(ip)
    records.append(rec)
    if vis_sample is None and (rec['signatures_detected'] > 0 or rec['stamps_detected'] > 0):
        vis_sample = (ip.name, bgr, det)

df_sig = pd.DataFrame(records)
with open(SIG_STAMP_REPORT_JSON, 'w', encoding='utf-8') as f:
    json.dump({'yolo_weights_detected': str(YOLO_WEIGHTS_PATH) if YOLO_WEIGHTS_PATH else None, 'results': records}, f, indent=2)

print(f"Screened {len(df_sig)} documents -> {SIG_STAMP_REPORT_JSON}")
if not df_sig.empty:
    display(df_sig.head(10))

if vis_sample is not None:
    fn, bgr, det = vis_sample
    canvas = bgr.copy()
    for d in det['signature_candidates']:
        x, y, bw, bh = d['bbox']
        cv2.rectangle(canvas, (x, y), (x + bw, y + bh), (0, 200, 0), 3)
    for d in det['stamp_candidates']:
        x, y, bw, bh = d['bbox']
        cv2.rectangle(canvas, (x, y), (x + bw, y + bh), (255, 0, 0), 3)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title(f'Signature (Green) & Stamp (Blue) Candidates — {fn}')
    plt.axis('off')
    plt.show()
